# FUSION AVEC CODE 

**Objectif** : Ajouter les codes associés à certaines modalités de variables 

Pour les varibales  ci dessous se trouvant  dans la base de travail qui sont des chaînes de caractère , on souhaite ajouter les codes associés à leur différentes modalités.
Pour ce faire, il faut utiliser un autre fichier excel où se trouve ces modalités et leurs codes pour chacune des variables.

Les variables en question sont : 

* Fonction
* Services
* Organismes
* Echelons
* Emplois
* Lieu d'affectation

L'objectif est de faire un fichier Excel propore avec les codes et libellés unique.  

## Chargement des packages necessaires

In [1]:
# Paramètres
import io
import pandas as pd
import boto3
import pandas as pd
import io
import pandas as pd
import unicodedata
import re
import numpy as np
import unicodedata, re
from pathlib import Path

## Chargement de la base de travail  

In [2]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/panel_solde_df_280725.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,DATE_NAISSANCE_CORRIGEE,ANNEE_NAISSANCE_CORRIGEE,MOIS_NAISSANCE_CORRIGEE,JOUR_NAISSANCE_CORRIGEE,AGE,AGE_VALIDE,SEXE_VALIDE,AGE_IMPUTE,SITUATION_MATRIMONIALE_RECODE,NBRE_ENFTS_IMPUTE
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,1939-01-01,1939.0,1.0,1.0,76.0,NaN,Masculin,41.0,Marié(e),0.0
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,1939-01-01,1939.0,1.0,1.0,76.0,NaN,Masculin,41.0,Célibataire,0.0
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,1924-01-01,1924.0,1.0,1.0,91.0,NaN,Masculin,41.0,Marié(e),0.0
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,1943-03-01,1943.0,3.0,1.0,71.0,NaN,Masculin,41.0,Marié(e),0.0
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,1945-12-01,1945.0,12.0,1.0,69.0,69.0,Masculin,69.0,Marié(e),2.0


In [3]:
# Paramètres S3
bucket_name = "admindataanstat"
object_key = "Solde/FICHIER_ANSTAT_CODE_2025.xlsx"  

# Télécharger le fichier depuis S3 dans un buffer mémoire
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(obj['Body'].read())

# Remettre le curseur au début
bytes_io.seek(0)

xlsx = pd.ExcelFile(bytes_io)
print("Feuilles disponibles :", xlsx.sheet_names)

Feuilles disponibles : ['lieu affectation', 'ORGANISME_OK', 'CODE_SITUATION_SOLDE', 'HISTORIQUE_ECHELLES_CORPS', 'ECHELLES_CORPS_ACTUEL', 'service', 'FONCTION', 'grade', 'LIBELLE POSTE']


# CREATION DES FONCTIONS NECESSAIRES

In [4]:
# Je défini une fonction que je nomme "normalize_label"
def normalize_label(s: str) -> str:
    """Minuscules, accents retirés, ponctuation uniformisée, espaces compressés."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""  # uniquement si c'est un vrai NaN
    s = str(s).strip()
    s = (s.replace("-", " ")
           .replace("’", "'")
           .replace("'", " ")
           .replace(".", " "))
    s = s.lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Je défini une fonction que je nomme "_pick_excel_engine" qui a pour objectif d’importer le module openpyxl.
# Si l’import réussit, la fonction renvoie "openpyxl"
# Si openpyxl n’est pas installé, Python va dans ce cas passer au cas exceptionnel. Essayer d’importer xlsxwriter.Si ça marche, il renvoie "xlsxwriter".

def _pick_excel_engine():
    try:
        import openpyxl  # noqa
        return "openpyxl"
    except Exception:
        try:
            import xlsxwriter  # noqa
            return "xlsxwriter"
        except Exception:
            raise RuntimeError("Installez 'openpyxl' ou 'xlsxwriter' pour écrire dans Excel.")

# Je défini une fonction que je nomme "write_sheet" qui a pour rôle d'écrire un DataFrame dans un fichier Excel,
# en remplaçant uniquement la feuille (sheet) spécifiée, sans effacer les autres feuilles du classeur

def write_sheet(df: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit/Remplace la feuille `sheet_name` dans `workbook_path`.
    - Si le classeur n'existe pas: création sans if_sheet_exists.
    - S'il existe: append + replace de la feuille.
    """
    engine = _pick_excel_engine()
    xlsx = Path(workbook_path)

    if engine == "openpyxl":
        if xlsx.exists():
            # append + replace la feuille
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
        else:
            # création (pas de if_sheet_exists en mode 'w')
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="w") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
    else:
        # xlsxwriter: on doit recharger les feuilles existantes pour les réécrire
        sheets = {}
        if xlsx.exists():
            try:
                with pd.ExcelFile(workbook_path) as xf:
                    for sn in xf.sheet_names:
                        sheets[sn] = xf.parse(sn)
            except Exception:
                sheets = {}
        sheets[sheet_name] = df
        with pd.ExcelWriter(workbook_path, engine="xlsxwriter") as xw:
            for sn, d in sheets.items():
                d.to_excel(xw, sheet_name=sn, index=False)

    print(f"✅ Feuille écrite : {sheet_name} → {workbook_path}")



## Chargement + Paramètres + Vérifs de base

In [5]:
# Définition d'une fonction pour lire et nettoyer les fichiers Excel
def read_reference(
    excel_source,
    sheet_name: str,
    required_cols=None,          # ex: ["CODE_FONCTION","LIBELLÉ_FONCTION"]
    alias_map: dict | None = None, # alias optionnel {col_source: col_cible}, appliqué UNIQUEMENT ici
    drop_title_substr: str | None = "TABLEAU",  # supprime les lignes dont la 1ʳᵉ colonne contient ce motif
    preview_rows: int = 5,
) -> pd.DataFrame:
    """
    Lit une feuille Excel, affiche un aperçu AVANT/APRÈS nettoyage et retourne le DataFrame propre.

    Nettoyage :
      - supprime lignes entièrement vides
      - supprime lignes “titre” si la 1ʳᵉ colonne contient `drop_title_substr` (ex: 'TABLEAU')
      - si besoin, promeut la 1ʳᵉ ligne en entête
      - applique un alias de colonnes (optionnel, UNIQUEMENT au chargement)
      - vérifie la présence des colonnes `required_cols` si fourni
    """

    # --- Lecture brute ---
    df_raw = pd.read_excel(excel_source, sheet_name=sheet_name)

    print(f"\n=== {sheet_name} | APERÇU BRUT (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df_raw.head(preview_rows))
    except Exception:
        print(df_raw.head(preview_rows).to_string(index=False))
    print(f"Colonnes BRUTES ({sheet_name}) :", df_raw.columns.tolist())
    print(f"Shape brut : {df_raw.shape}")

    # --- Copie de travail ---
    df = df_raw.copy()

    # 1) Supprimer les lignes entièrement vides
    df = df.dropna(how="all")

    # 2) Supprimer les lignes de titre si la 1ʳᵉ colonne les contient (ex: 'TABLEAU')
    if drop_title_substr is not None and not df.empty:
        first_col = df.columns[0]
        mask_title = df[first_col].astype(str).str.contains(drop_title_substr, case=False, na=False)
        df = df.loc[~mask_title].copy()

    # 3) Re-détecter l'entête si nécessaire (heuristique simple)
    def _looks_like_header(row_vals, required, alias):
        vals = [str(v).strip() for v in row_vals]
        if required and any(c in vals for c in required):
            return True
        if alias and any(src in vals for src in alias.keys()):
            return True
        if any("unnamed" in v.lower() for v in vals):
            return False
        if any(("code" in v.lower()) or ("lib" in v.lower()) for v in vals):
            return True
        return False

    if not df.empty:
        candidate = df.iloc[0].tolist()
        if _looks_like_header(candidate, required_cols, alias_map):
            df.columns = [str(v).strip() for v in candidate]
            df = df.drop(df.index[0]).reset_index(drop=True)

    # 4) Alias de colonnes (optionnel, seulement au chargement)
    if alias_map:
        df = df.rename(columns=alias_map)

    # 5) Vérifier les colonnes requises (si fourni)
    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print("\n⚠️ Colonnes manquantes après nettoyage/alias :", missing)
            print("Colonnes disponibles :", df.columns.tolist())
            raise ValueError(f"Colonnes manquantes : {missing}")

    # 6) Aperçu après nettoyage
    print(f"\n=== {sheet_name} | APRÈS NETTOYAGE (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df.head(preview_rows))
    except Exception:
        print(df.head(preview_rows).to_string(index=False))
    print(f"Colonnes NETTOYÉES ({sheet_name}) :", df.columns.tolist())
    print(f"Shape nettoyé : {df.shape}")

    return df.reset_index(drop=True)


def check_code_to_label_uniqueness(df_ref: pd.DataFrame, var_code_ext: str, var_lib_ext: str) -> pd.Series:
    """
    Vérifie que chaque code (var_code_ext) pointe vers un seul libellé (var_lib_ext).
    Retourne la série des codes non-uniques (counts).
    """
    verif_code = df_ref.groupby(var_code_ext)[var_lib_ext].nunique()
    bad = verif_code[verif_code > 1]
    if bad.empty:
        print("✅ Chaque code est associé à un seul libellé.")
    else:
        print("❌ Codes associés à plusieurs libellés :")
        print(bad)
    return bad

def ensure_normalized_column(df_ref: pd.DataFrame, var_lib_ext: str, var_norm_ext: str | None = None) -> str:
    """
    Ajoute la colonne normalisée (var_norm_ext) si absente. Retourne son nom.
    """
    var_norm_ext = var_norm_ext or f"{var_lib_ext}_NORM"
    if var_norm_ext not in df_ref.columns:
        df_ref[var_norm_ext] = df_ref[var_lib_ext].apply(normalize_label)
    return var_norm_ext

def detect_normalized_duplicates(df_ref: pd.DataFrame,
                                 var_norm_ext: str,
                                 var_lib_ext: str,
                                 var_code_ext: str) -> pd.DataFrame:
    """
    Détecte les doublons après normalisation et retourne toutes les lignes uniques concernées.

    Paramètres
    ----------
    df_ref : pd.DataFrame
        Table de référence contenant les codes et libellés normalisés.
    var_norm_ext : str
        Nom de la colonne contenant les libellés normalisés.
    var_lib_ext : str
        Nom de la colonne contenant les libellés originaux.
    var_code_ext : str
        Nom de la colonne contenant les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes uniques dont la clé normalisée apparaît plusieurs fois.
    """
    # Compter combien de fois chaque clé normalisée apparaît
    counts = df_ref[var_norm_ext].value_counts()
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code_ext, var_lib_ext, var_norm_ext])

    # Filtrer toutes les lignes concernées
    dup_rows = df_ref[df_ref[var_norm_ext].isin(repeated_keys)].copy()

    # Garder uniquement les lignes uniques (code + libellé + normalisé)
    dup_rows = dup_rows.drop_duplicates(subset=[var_code_ext, var_lib_ext, var_norm_ext])

    # Message d’alerte clair
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes uniques concernées.")

    return dup_rows

In [6]:
def report_duplicates(df: pd.DataFrame, var_norm: str, var_lib: str, var_code: str) -> pd.DataFrame:
    """
    Affiche la liste exhaustive des doublons après normalisation et retourne toutes les lignes concernées.
    
    Paramètres
    ----------
    df : pd.DataFrame
        Table contenant les libellés et codes.
    var_norm : str
        Colonne avec les libellés normalisés (clé de regroupement).
    var_lib : str
        Colonne avec les libellés originaux.
    var_code : str
        Colonne avec les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes du DataFrame où une même clé normalisée apparaît plusieurs fois.
    """

    # --- 1) Compter toutes les occurrences de la clé normalisée ---
    counts = df[var_norm].value_counts()

    # --- 2) Sélectionner les clés répétées ---
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code, var_lib, var_norm])

    # --- 3) Filtrer toutes les lignes concernées ---
    dup_rows = df[df[var_norm].isin(repeated_keys)].copy()

    # --- 4) Message d’alerte ---
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes concernées.")

    # --- 5) Affichage des colonnes essentielles pour l’audit ---
    try:
        from IPython.display import display
        display(dup_rows[[var_code, var_lib, var_norm]])
    except Exception:
        print(dup_rows[[var_code, var_lib, var_norm]].to_string(index=False))

    # --- 6) Retourner le DataFrame complet ---
    return dup_rows

## Construction du mapping (1 code par libellé normalisé)

In [7]:
# La fonction "build_random_mapping" a pour but de créer une association unique entre des codes et les libellés normalisés dans un DataFrame,
# en gérant les ambiguïtés et en ajoutant d'autres informations

def build_random_mapping(
    df_ref: pd.DataFrame, # Le DataFrame source contenant les données (codes, libellés brutww, libellés normalisés)
    var_norm_ext: str, # Nom de la colonne "libellé normalisé" sur laquelle on veut créer le mapping
    var_code_ext: str, # Nom de la colonne "code" qui sera associé au libellé normalisé
    var_lib_ext: str, # Nom de la colonne "libellé brut" correspondant au libellé de départ avant traitement
    seed: int = 123 # Seed pour rendre le tirage aléatoire reproductible
) -> pd.DataFrame:
    """
    Construit un mapping 1 code / clé normalisée :
      - choisit aléatoirement un code quand plusieurs existent (traçable par seed),
      - ajoute les colonnes d’audit : EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES.
    Sortie : DataFrame avec colonnes [var_lib_ext, var_norm_ext, var_code_ext,
                                     EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES, CODE_CHOISI]
    """
    df = df_ref.copy() # On crée une copie du DataFrame d’origine
    # infos par clé
    codes_list = (df.groupby(var_norm_ext)[var_code_ext]
                    .apply(lambda s: sorted(map(str, pd.unique(s))))
                    .rename("CODES_LIST")) # créer pour chaque libellé normalisé (var_norm_ext) la liste des codes uniques associés (var_code_ext).
    codes_info = codes_list.to_frame()
    codes_info["NB_CODES_POSSIBLES"] = codes_info["CODES_LIST"].str.len() # Compte le nombre de codes possibles pour chaque libellé normalisé
    codes_info["EST_AMBIGU"] = codes_info["NB_CODES_POSSIBLES"].gt(1) #Crée une colonne booléenne indiquant si le libellé normalisé est associé à plusieurs codes.1" → True si plus d’un code, False sinon
    codes_info["CODES_POSSIBLES"] = codes_info["CODES_LIST"].apply(lambda L: ", ".join(L))

    # tirage aléatoire d'un code par clé normalisée
    pairs_uniques = df[[var_norm_ext, var_code_ext]].drop_duplicates() # On prend seulement les colonnes libellé normalisé (var_norm_ext) et code (var_code_ext) du DataFrame. Ensuite, on supprime les lignes identiques pour ne garder qu’une seule occurrence par combinaison
    selection = (pairs_uniques.groupby(var_norm_ext, group_keys=False)
                 .apply(lambda g: g.sample(n=1, random_state=seed))
                 .reset_index(drop=True)
                 .rename(columns={var_code_ext: "CODE_CHOISI"}))
    # groupby(var_norm_ext, group_keys=False) : On regroupe les lignes par libellé normalisé et chaque groupe contient tous les codes possibles pour ce libellé.
    # .apply(lambda g: g.sample(n=1, random_state=seed)) : pour chaque libellé, on choisit aléatoirement un code parmi les codes possibles. random_state=seed permet que le tirage soit répétable : même code choisi si on réexécute le programme avec le même seed. reset_index(drop=True) : Réinitialise l’index du DataFrame résultant pour avoir un index simple allant de 0 à n-1.
    # .rename(columns={var_code_ext: "CODE_CHOISI"}) : renomme la colonne du code choisi pour clarifier que c’est celui sélectionné aléatoirement.

    # mapping final = libellé brut x clé norm + code choisi + audit d’ambiguïté
    mapping = (df[[var_lib_ext, var_norm_ext]]
               .drop_duplicates() 
               # On récupère les colonnes libellé brut (var_lib_ext) et libellé normalisé (var_norm_ext). Ensuite, on supprime les doublons pour avoir une seule ligne par combinaison libellé brut / libellé normalisé.
               
               .merge(selection, on=var_norm_ext, how="left") 
               # On ajoute la colonne CODE_CHOISI à chaque libellé normalisé. un code choisi aléatoirement par libellé normalisé.
               
               .merge(codes_info.drop(columns=["CODES_LIST"]), left_on=var_norm_ext, right_index=True, how="left"))
                # On ajoute les informations d’audit (nombre de codes possibles, liste des codes, indicateur d’ambiguïté) depuis codes_info. on supprime la liste brute des codes pour ne garder que les colonnes utiles pour l’audit.
    mapping[var_code_ext] = mapping["CODE_CHOISI"]

    cols = [var_lib_ext, var_norm_ext, var_code_ext, "CODE_CHOISI", "EST_AMBIGU",
            "NB_CODES_POSSIBLES", "CODES_POSSIBLES"]
    mapping = mapping[cols].sort_values([ "EST_AMBIGU", var_norm_ext, var_lib_ext ],
                                        ascending=[False, True, True])
    return mapping

def export_mapping(mapping: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit UNIQUEMENT la feuille de mapping (pas de feuille ‘Ambigus’) ;
    la feuille contient les colonnes d’audit ‘EST_AMBIGU’, ‘CODES_POSSIBLES’, etc.
    """
    write_sheet(mapping, workbook_path, sheet_name)


## Fusion du mapping dans le panel + bilan des non appariés

In [8]:
def merge_mapping_into_panel(panel_df: pd.DataFrame,
                             mapping: pd.DataFrame,
                             var_norm_panel: str, var_norm_ext: str,
                             var_code_panel: str, var_code_ext: str,
                             var_lib_panel: str):
    """
    Met à jour les codes du panel à partir du mapping via les libellés normalisés.
    Le code est récupéré directement du mapping et prend le nom de la colonne du panel.
    """
    # 1️⃣ Copie du panel
    df = panel_df.copy()

    # 2️⃣ Créer la colonne normalisée dans le panel si absente
    if var_norm_panel not in df.columns:
        df[var_norm_panel] = df[var_lib_panel].apply(normalize_label)

    # 3️⃣ Préparer le mapping (suppression doublons)
    mapping_clean = mapping[[var_norm_ext, var_code_ext]].drop_duplicates()
    mapping_clean[var_code_ext] = mapping_clean[var_code_ext].astype("string").fillna("NA")

    # 4️⃣ Merge sur la clé normalisée et renommer la colonne du code pour matcher le panel
    out = df.merge(mapping_clean,
                   left_on=var_norm_panel,
                   right_on=var_norm_ext,
                   how="left")\
            .rename(columns={var_code_ext: var_code_panel})

    # 5️⃣ Supprimer les colonnes techniques issues du merge
    out = out.drop(columns=[var_norm_ext], errors="ignore")

    # 6️⃣ Statistiques
    nb_rens = int(out[var_code_panel].notna().sum())
    nb_miss = int(out[var_code_panel].isna().sum())
    print(f"Codes renseignés dans {var_code_panel} : {nb_rens}")
    print(f"Codes manquants dans {var_code_panel} : {nb_miss}")

    return out, nb_rens, nb_miss


## Similarité sur les non appariés (top 1 par clé)

In [9]:
#  la fonction "list_unmatched_norms" liste toutes les libellés normalisés (var_norm_panel) dans le panel qui n’ont pas de code associé (var_code_panel vide ou NA).
# Cela permet de détecter les éléments du panel qui n’ont pas pu être appariés avec le mapping.


''' Définition d’une fonction qui prend : 
(1) panel_df  qui est le tableau de données principal (panel)
(2) var_code_panel : la colonne du code qui peut être manquante (NA)
(3) var_norm_panel : la colonne du libellé normalisé '''

def list_unmatched_norms(panel_df: pd.DataFrame, var_code_panel: str, var_norm_panel: str) -> pd.DataFrame:
    """
    Retourne les libellés normalisés non appariés avec le nombre de lignes correspondantes,
    y compris les libellés vides ou entièrement blancs.
    """
    # Sélection des lignes où le code est NA
    s = panel_df.loc[panel_df[var_code_panel].isna(), var_norm_panel].astype(str)

    # Nettoyer les chaînes (supprimer espaces en début/fin)
    s = s.str.strip()

    # Compter les occurrences de chaque libellé, y compris les vides
    counts = s.value_counts(dropna=False).reset_index()
    counts.columns = [var_norm_panel, "COUNT"]

    if counts.empty:
        print("✅ Tous les libellés ont été appariés, aucun non apparié.")
    else:
        total_non_apparies = counts["COUNT"].sum()
        print(f"⚠️ {len(counts)} libellés non appariés trouvés (total lignes concernées : {total_non_apparies}) :")
        try:
            from IPython.display import display
            display(counts)
        except Exception:
            print(counts.to_string(index=False))

    return counts
        
# Cette fonction "best_suggestions" cherche pour chaque libellé non apparié le meilleur candidat dans le référentiel
def best_suggestions(panel_df: pd.DataFrame, mapping: pd.DataFrame,
                     var_norm_panel: str, var_norm_ext: str,
                     var_lib_ext: str, var_code_ext: str,
                     var_code_panel: str,
                     top_n: int = 20,
                     min_score_show: int = 0) -> pd.DataFrame:
    """
    Pour chaque libellé normalisé non apparié dans le panel, calcule le MEILLEUR candidat
    du référentiel (score de similarité). Affiche le top et retourne un DataFrame suggestions_best.

    Paramètres
    ----------
    panel_df : pd.DataFrame
        DataFrame du panel principal (contenant les codes éventuellement manquants)
    mapping : pd.DataFrame
        Référentiel avec les codes et libellés normalisés
    var_norm_panel : str
        Nom de la colonne des libellés normalisés dans le panel
    var_norm_ext : str
        Nom de la colonne des libellés normalisés dans le référentiel
    var_lib_ext : str
        Nom de la colonne des libellés originaux dans le référentiel
    var_code_ext : str
        Nom de la colonne des codes dans le référentiel
    var_code_panel : str
        Nom de la colonne des codes dans le panel
    top_n : int
        Nombre maximum de suggestions à afficher
    min_score_show : int
        Seuil minimal pour afficher les suggestions (score de similarité)
    
    Retour
    ------
    pd.DataFrame
        DataFrame avec les meilleures suggestions pour chaque libellé non apparié.
        Colonnes : [f"{var_norm_panel}_NON_APPARIE", f"MATCH_{var_norm_ext}", "MATCH_SCORE",
                    var_lib_ext, var_code_ext]
    """

    # --- 1) Extraire les libellés non appariés dans le panel ---
    # Utilise list_unmatched_norms qui retourne COUNT, mais ici on prend uniquement la liste
    non_apparies_df = list_unmatched_norms(panel_df, var_code_panel, var_norm_panel)
    non_apparies = non_apparies_df[var_norm_panel].tolist()  # conversion en liste simple

    # --- 2) Préparer la liste des clés normalisées du référentiel ---
    ref_norms = mapping[var_norm_ext].dropna().astype(str).str.strip()
    ref_norms = ref_norms[ref_norms.ne("")].unique().tolist()  # enlever les vides et doublons

    # --- 3) Déterminer le moteur de similarité ---
    use_rapidfuzz = False
    try:
        from rapidfuzz import process, fuzz  # type: ignore
        use_rapidfuzz = True  # RapidFuzz disponible pour calculer la similarité rapidement
    except Exception:
        from difflib import SequenceMatcher  # fallback si RapidFuzz non installé

    # --- 4) Fonction interne pour calculer le meilleur match d’un libellé ---
    def _best_match(q: str):
        if not q or q.strip() == "":
            return None  # pas de texte → pas de match possible
        if use_rapidfuzz:
            # renvoie (libellé normalisé correspondant, score, index)
            return process.extractOne(q, ref_norms, scorer=fuzz.token_set_ratio)
        else:
            # fallback : utiliser SequenceMatcher pour comparer chaque libellé
            best_score, best_idx = -1, None
            for idx, rn in enumerate(ref_norms):
                sc = int(SequenceMatcher(None, q, rn).ratio() * 100)
                if sc > best_score:
                    best_score, best_idx = sc, idx
            return (ref_norms[best_idx], best_score, best_idx) if best_idx is not None else None

    # --- 5) Chercher le meilleur candidat pour chaque libellé non apparié ---
    rows = []
    for qn in non_apparies:
        res = _best_match(qn)

        if res is None:
            # Cas où le libellé est vide ou aucun match trouvé
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": "AUCUNE_SUGGESTION",
                "MATCH_SCORE": 0,
                var_lib_ext: "Aucune suggestion possible",
                var_code_ext: None
            })
            continue

        matched_norm, score, _ = res

        # récupérer la ligne correspondante dans le mapping (libellé et code)
        # si plusieurs correspondances, on prend la première pour la stabilité
        cand = mapping.loc[mapping[var_norm_ext] == matched_norm, [var_lib_ext, var_code_ext]].head(1)
        if cand.empty:
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": matched_norm,
                "MATCH_SCORE": int(score),
                var_lib_ext: "Non trouvé dans mapping",
                var_code_ext: None
            })
            continue

        lib_val = cand.iloc[0][var_lib_ext]
        code_val = cand.iloc[0][var_code_ext]

        # construire une ligne pour le DataFrame des suggestions
        rows.append({
            f"{var_norm_panel}_NON_APPARIE": qn,
            f"MATCH_{var_norm_ext}": matched_norm,
            "MATCH_SCORE": int(score),
            var_lib_ext: lib_val,
            var_code_ext: code_val,
        })

    # --- 6) Colonnes attendues ---
    expected_cols = [
        f"{var_norm_panel}_NON_APPARIE",
        f"MATCH_{var_norm_ext}",
        "MATCH_SCORE",
        var_lib_ext,
        var_code_ext,
    ]

    # --- 7) Création du DataFrame final ---
    suggestions_best = pd.DataFrame(rows)

    # Trier par score puis par libellé pour afficher les meilleures suggestions en haut
    if not suggestions_best.empty:
        suggestions_best = suggestions_best.sort_values(
            ["MATCH_SCORE", var_lib_ext], ascending=[False, True]
        )[expected_cols]
        if min_score_show > 0:
            to_show = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score_show].head(top_n)
        else:
            to_show = suggestions_best.head(top_n)
    else:
        # DataFrame vide structuré avec les colonnes attendues
        suggestions_best = pd.DataFrame(columns=expected_cols)
        to_show = suggestions_best

    # --- 8) Affichage Jupyter-friendly ---
    try:
        from IPython.display import display
        display(to_show)
    except Exception:
        print(to_show.to_string(index=False))

    return suggestions_best


## Application des remplacements (manuel OU automatique)

In [10]:
def apply_choices(panel_df: pd.DataFrame,
                  var_code_panel: str, var_norm_panel: str,
                  choices_map: dict[str, str],
                  var_cle_panel: str, var_lib_panel: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Applique un dictionnaire {clé_norm_panel -> code}, uniquement là où le code est NA.
    Renvoie (panel_df_mis_a_jour, audit_df).
    """
    df = panel_df.copy()
    # Masque des lignes où le code est manquant
    fill_mask = df[var_code_panel].isna()

    # Colonne temporaire pour stocker les suggestions de code
    sugg_col = f"{var_code_panel}_SUGG"
    df.loc[fill_mask, sugg_col] = df.loc[fill_mask, var_norm_panel].map(choices_map)

    # Masque des lignes où : var_code_panel est encore NA mais une suggestion a été trouvée dans sugg_col
    m_apply = df[var_code_panel].isna() & df[sugg_col].notna()
    
    # Création d’une colonne BEFORE pour garder une trace de l’état initial des codes avant mise à jour.
    before_col = f"{var_code_panel}_BEFORE" 
    df[before_col] = df[var_code_panel]
    

    # Remplit les codes manquants avec les valeurs suggérées.
    df.loc[m_apply, var_code_panel] = df.loc[m_apply, sugg_col] 
    
    '''
    Crée un DataFrame d’audit qui contient :

           - la clé et le libellé d’origine
           - le code avant mise à jour
           - le code après mise à jour (AFTER)

     Cela permet de suivre exactement quelles lignes ont été modifiées.
     '''

    audit = (df.loc[m_apply, [var_cle_panel, var_lib_panel, var_norm_panel, before_col, var_code_panel]]
               .copy()
               .rename(columns={var_code_panel: f"{var_code_panel}_AFTER"}))

    print(f"✔ Lignes mises à jour : {len(audit)}")
    df.drop(columns=[sugg_col], errors="ignore", inplace=True)
    return df, audit



In [11]:
"""
Automatiser l’application des suggestions de codes sur le panel uniquement pour les suggestions avec un score de similarité supérieur à un seuil min_score.
Elle utilise la fonction apply_choices pour remplir les codes manquants.
Définition de la fonction avec les paramètres :

     - panel_df : le tableau de données principal où on veut remplir les codes.
     - suggestions_best : le DataFrame contenant les suggestions de codes avec score de similarité.
     - var_code_panel : colonne du code dans le panel (celle à remplir).
     - var_norm_panel : colonne du libellé normalisé dans le panel.
     - var_cle_panel : colonne contenant une clé d’identification unique pour chaque ligne.
     - var_lib_panel : colonne du libellé original dans le panel.
     - var_code_ext : colonne du code dans le référentiel de suggestions.
     - min_score : seuil minimal pour appliquer automatiquement la suggestion (défaut 70).

La fonction retourne un tuple (panel_df_mis_a_jour, audit_df).
"""
def auto_apply_by_score(panel_df: pd.DataFrame,
                        suggestions_best: pd.DataFrame,
                        var_code_panel: str, var_norm_panel: str,
                        var_cle_panel: str, var_lib_panel: str,
                        var_code_ext: str,
                        min_score: int = 70) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construit choices_map pour toutes les suggestions avec score ≥ min_score, et applique.
    """
    if suggestions_best.empty: # Vérifie si le DataFrame des suggestions est vide.
        print("INFO : pas de suggestions à appliquer.")
        return panel_df.copy(), pd.DataFrame()  # Si oui : on affiche un message, et on retourne une copie du panel inchangé et un DataFrame d’audit vide.

    # nom de la colonne 'libelle_norm_non_apparie' dans suggestions_best
    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("La table suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # Filtre les suggestions pour ne garder que celles dont le score de similarité est supérieur ou égal à min_score. Seules ces suggestions seront appliquées
    ok = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score]

    # Construit un dictionnaire clé → code à partir des suggestions filtrées : la clé est le libellé normalisé non apparié dans le panel (norm_non_app_col). la valeur est le code correspondant dans le référentiel (var_code_ext).
    # Ce dictionnaire sera utilisé par apply_choices pour remplir automatiquement les codes.
    choices_map = dict(zip(ok[norm_non_app_col], ok[var_code_ext]))

    return apply_choices(panel_df, var_code_panel, var_norm_panel, choices_map, var_cle_panel, var_lib_panel)


In [12]:
def enrich_mapping_with_choices(mapping_final: pd.DataFrame,
                                suggestions_best: pd.DataFrame,
                                var_norm_panel: str,
                                var_norm_ext: str, var_code_ext: str, var_lib_ext: str,
                                choices_map: dict[str, str]) -> pd.DataFrame:
    """
    Ajoute dans le mapping les alias normalisés retenus (clé_norm_panel -> code),
    en remplaçant l’éventuelle ancienne ligne pour cette clé.
    """
    if suggestions_best.empty or not choices_map:
        print("INFO : rien à enrichir (pas de suggestions ou pas de choix).")
        return mapping_final

    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # ne garder que les suggestions dont la clé est choisie
    sel = suggestions_best[suggestions_best[norm_non_app_col].isin(list(choices_map.keys()))].copy()

    # remplacer le code par celui imposé par choices_map (au cas où)
    sel[var_code_ext] = sel[norm_non_app_col].map(choices_map)

    add = (sel[[norm_non_app_col, var_code_ext, var_lib_ext]]
             .drop_duplicates()
             .rename(columns={norm_non_app_col: var_norm_ext}))

    need_cols = {var_norm_ext, var_lib_ext, var_code_ext}
    missing = [c for c in need_cols if c not in mapping_final.columns]
    if missing:
        raise ValueError(f"mapping_final doit contenir {need_cols}, manquant : {missing}")

    before = len(mapping_final)
    # retire les anciennes lignes pour ces clés
    mapping_final = mapping_final[~mapping_final[var_norm_ext].isin(add[var_norm_ext])]
    # concatène + dédoublonne
    mapping_final = (pd.concat([mapping_final, add], ignore_index=True)
                     .drop_duplicates(subset=[var_norm_ext, var_code_ext]))
    print(f"🔁 Mapping enrichi : +{len(mapping_final)-before} lignes (net).")
    return mapping_final


## FONCTION

In [13]:
# Lire la feuille FONCTION dans notre fichier Excel de référence
fichier_ref = "FONCTION"
df_ref = read_reference(bytes_io, fichier_ref)


=== FONCTION | APERÇU BRUT (top 5) ===


,CODE_FONCTION,LIBELLÉ_FONCTION
0,0,
1,1,Personnel Médical Contractuel exerçant des soi...
2,2,Indemnité spécifique au Directeur de Cabinet M...
3,4,Indemnité du Chargé de Mission ou Secrétaire P...
4,5,Moniteur des Sciences Fondamentales


Colonnes BRUTES (FONCTION) : ['CODE_FONCTION', 'LIBELLÉ_FONCTION']
Shape brut : (160, 2)

=== FONCTION | APRÈS NETTOYAGE (top 5) ===


,CODE_FONCTION,LIBELLÉ_FONCTION
0,0,
1,1,Personnel Médical Contractuel exerçant des soi...
2,2,Indemnité spécifique au Directeur de Cabinet M...
3,4,Indemnité du Chargé de Mission ou Secrétaire P...
4,5,Moniteur des Sciences Fondamentales


Colonnes NETTOYÉES (FONCTION) : ['CODE_FONCTION', 'LIBELLÉ_FONCTION']
Shape nettoyé : (160, 2)


In [14]:
# Données panel solde
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE'],
      dtype='object')

**Définition des paramètres**

In [15]:
nm_feuille = "FONCTION" # nom de la feuille Excel à charger
var_code_ext = "CODE_FONCTION" # la colonne de code dans la table externe (fichier excel)
var_lib_ext  = "LIBELLÉ_FONCTION" #la colonne de libellé dans la table externe (fichier Excel)
var_cle_ext  = var_lib_ext # on choisit le libellé comme clé principale côté référence (fichier Excel)

var_cle_panel  = "FONCTION" # clé utilisée dans le fichier panel (base principale)
var_code_panel = "CODE_FONCTION" # colonne de code dans le panel
var_lib_panel  = "FONCTION" # colonne de libellé dans le panel

# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"  # nom de la variable de libellé normalisé dans la table de référence (fichier Excel)
var_norm_panel = f"{var_cle_panel}_NORM"  # nom de la variable de libellé normalisé dans le panel

**1) Vérification**

In [16]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 5 clés répétées, 11 lignes uniques concernées.


In [17]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 5 clés répétées, 11 lignes concernées.


,CODE_FONCTION,LIBELLÉ_FONCTION,LIBELLÉ_FONCTION_NORM
26,48,agent comptable du trésor,agent comptable du tresor
27,49,Trésorier Général,tresorier general
54,87,Agent Comptable du Trésor,agent comptable du tresor
55,88,Trésorier,tresorier
56,89,Trésorier,tresorier
94,665,receveur des impots,receveur des impots
102,673,Receveur d'Enregistrement,receveur d enregistrement
103,674,Trésorier Général,tresorier general
127,700,receveur d'enregistrement,receveur d enregistrement
140,763,trésorier général,tresorier general


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [18]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : FONCTION → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [19]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans CODE_FONCTION : 15627891
Codes manquants dans CODE_FONCTION : 72


**4) Similarité pour les non appariés (top 1 par clé)**

In [20]:
suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

⚠️ 1 libellés non appariés trouvés (total lignes concernées : 72) :


,FONCTION_NORM,COUNT
0,tresorier charge d etudes g4,72


,FONCTION_NORM_NON_APPARIE,MATCH_LIBELLÉ_FONCTION_NORM,MATCH_SCORE,LIBELLÉ_FONCTION,CODE_FONCTION
0,tresorier charge d etudes g4,tresorier charge etudes g4,96,Trésorier Chargé études G4,797


**5a)Application MANUELLE (tu fournis tes choix)**

In [21]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "tresorier charge d etudes g4": "797"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 72


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [22]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)

🔁 Mapping enrichi : +-3 lignes (net).


In [23]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'CODE_FONCTION_BEFORE'],
      dtype='object')

In [24]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_FONCTION_BEFORE'] != panel_solde_df['CODE_FONCTION']) | \
            (panel_solde_df['CODE_FONCTION_BEFORE'].isna() & panel_solde_df['CODE_FONCTION'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_FONCTION_BEFORE', 'CODE_FONCTION', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

Nombre d'observations modifiées ou complétées : 72


,CODE_FONCTION_BEFORE,CODE_FONCTION,FONCTION_NORM
30252,<NA>,797,tresorier charge d etudes g4


In [25]:
# Supprime la colonne CODE_FONCTION_BEFORE
panel_solde_df= panel_solde_df.drop(columns=['CODE_FONCTION_BEFORE'])

In [26]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION'],
      dtype='object')

## SERVICES

In [27]:
# Lire la feuille SERVICE dans notre fichier Excel de référence
fichier_ref = "service"
df_ref = read_reference(bytes_io, fichier_ref)


=== service | APERÇU BRUT (top 5) ===


,CODE_SERVICE,LIBELLÉ_SERVICE
0,0200,A affecter
1,0205,Insp. Gén. d Etat
2,0210,Présidence de la République
3,02GM,Grand Médiateur
4,02RI,Relations avec les Institutions


Colonnes BRUTES (service) : ['CODE_SERVICE', 'LIBELLÉ_SERVICE']
Shape brut : (2241, 2)

=== service | APRÈS NETTOYAGE (top 5) ===


,CODE_SERVICE,LIBELLÉ_SERVICE
0,0200,A affecter
1,0205,Insp. Gén. d Etat
2,0210,Présidence de la République
3,02GM,Grand Médiateur
4,02RI,Relations avec les Institutions


Colonnes NETTOYÉES (service) : ['CODE_SERVICE', 'LIBELLÉ_SERVICE']
Shape nettoyé : (2241, 2)


In [28]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION'],
      dtype='object')

**Paramètres**

In [29]:
nm_feuille = "SERVICE" # nom de la feuille 
var_code_ext = "CODE_SERVICE"
var_lib_ext  = "LIBELLÉ_SERVICE"
var_cle_ext  = var_lib_ext

var_cle_panel  = "SERVICE"
var_code_panel = "CODE_SERVICE"
var_lib_panel  = "SERVICE"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

**1) Vérification**

In [30]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 180 clés répétées, 539 lignes uniques concernées.


In [31]:
# Afficher la liste des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 180 clés répétées, 539 lignes concernées.


,CODE_SERVICE,LIBELLÉ_SERVICE,LIBELLÉ_SERVICE_NORM
0,0200,A affecter,a affecter
5,0310,Cabinet,cabinet
6,0311,D A A F,d a a f
7,0400,A affecter,a affecter
8,0410,Cabinet Hotel,cabinet hotel
...,...,...,...
2231,OP10,O I P R,o i p r
2234,PL10,PSP LIQUIDATION,psp liquidation
2235,PS10,PSP LIQUIDATION,psp liquidation
2237,SG10,SOGEPIE,sogepie


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [32]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : SERVICE → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [33]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans CODE_SERVICE : 15470956
Codes manquants dans CODE_SERVICE : 157007


**4) Similarité pour les non appariés (top 1 par clé)**

In [34]:
suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

⚠️ 4 libellés non appariés trouvés (total lignes concernées : 157007) :


,SERVICE_NORM,COUNT
0,e p h d,150505
1,ppa urc,5972
2,delegation de paris unesco,523
3,,7


,SERVICE_NORM_NON_APPARIE,MATCH_LIBELLÉ_SERVICE_NORM,MATCH_SCORE,LIBELLÉ_SERVICE,CODE_SERVICE
2,delegation de paris unesco,delegation paris unesco,93,Délégation Paris Unesco,5950
1,ppa urc,ppa_urc,85,PPA_URC,38Q5
0,e p h d,d p s d,71,D P S D,52DP
3,,AUCUNE_SUGGESTION,0,Aucune suggestion possible,None


**5a)Application MANUELLE (tu fournis tes choix)**

In [35]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "delegation de paris unesco": "5950","ppa urc": "38Q5"
    
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 6495


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [36]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


🔁 Mapping enrichi : +-58 lignes (net).


In [37]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'CODE_SERVICE_BEFORE'],
      dtype='ob

In [38]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_SERVICE_BEFORE'] != panel_solde_df['CODE_SERVICE']) | \
            (panel_solde_df['CODE_SERVICE_BEFORE'].isna() & panel_solde_df['CODE_SERVICE'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_SERVICE_BEFORE', 'CODE_SERVICE', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

Nombre d'observations modifiées ou complétées : 6495


,CODE_SERVICE_BEFORE,CODE_SERVICE,SERVICE_NORM
334,<NA>,5950,delegation de paris unesco
2499484,<NA>,38Q5,ppa urc


In [39]:
# Supprimer la variable CODE_SERVICE_BEFORE
panel_solde_df=panel_solde_df.drop(columns=["CODE_SERVICE_BEFORE"])

In [40]:
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE'],
      dtype='object')

## ORGANISMES

In [41]:
# Lecture de la feuille ORGANISME du fichier de reférence (Excel)
fichier_ref = "ORGANISME_OK"
df_ref = read_reference(bytes_io, fichier_ref)


=== ORGANISME_OK | APERÇU BRUT (top 5) ===


,CODE_ORGANISME,LIBELLÉ_ORGANISME,CODE_ÉTABLISSEMENT,CODE_ACCT
0,00,En Attente d'Affectation,1,NaN
1,02,Présidence (Budget Général),1,NaN
2,03,Min Réconciliation Nationale,1,NaN
3,04,Cour Suprême,1,NaN
4,05,Grande Chancellerie,1,NaN


Colonnes BRUTES (ORGANISME_OK) : ['CODE_ORGANISME', 'LIBELLÉ_ORGANISME', 'CODE_ÉTABLISSEMENT', 'CODE_ACCT']
Shape brut : (210, 4)

=== ORGANISME_OK | APRÈS NETTOYAGE (top 5) ===


,CODE_ORGANISME,LIBELLÉ_ORGANISME,CODE_ÉTABLISSEMENT,CODE_ACCT
0,00,En Attente d'Affectation,1,NaN
1,02,Présidence (Budget Général),1,NaN
2,03,Min Réconciliation Nationale,1,NaN
3,04,Cour Suprême,1,NaN
4,05,Grande Chancellerie,1,NaN


Colonnes NETTOYÉES (ORGANISME_OK) : ['CODE_ORGANISME', 'LIBELLÉ_ORGANISME', 'CODE_ÉTABLISSEMENT', 'CODE_ACCT']
Shape nettoyé : (210, 4)


In [42]:
"""
Renommer la variable CODE_ORGANISME en ID_ORGANISME pour éviter toutes confusion 
avec la base de données panel où il y a déjà la variable CODE_ORGANISME
"""
df_ref = df_ref.rename({"CODE_ORGANISME": "ID_ORGANISME"}, axis=1)

In [43]:
# Liste des variables de la base de données Panel
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE'],
      dtype='object')

**Paramètres**

In [44]:
nm_feuille = "ORGANISME" # nom de la feuille 
var_code_ext = "ID_ORGANISME"
var_lib_ext  = "LIBELLÉ_ORGANISME"
var_cle_ext  = var_lib_ext

var_cle_panel  = "ORGANISME"
var_code_panel = "ID_ORGANISME"
var_lib_panel  = "ORGANISME"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

**1) Vérification**

In [45]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 4 clés répétées, 16 lignes uniques concernées.


In [46]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 4 clés répétées, 16 lignes concernées.


,ID_ORGANISME,LIBELLÉ_ORGANISME,LIBELLÉ_ORGANISME_NORM
59,60,xxxx,xxxx
60,61,xxxx,xxxx
61,63,xxxx,xxxx
62,64,xxxx,xxxx
64,81,xxxx,xxxx
65,82,xxxx,xxxx
66,83,xxxx,xxxx
67,84,xxxx,xxxx
68,85,xxxx,xxxx
69,86,xxxx,xxxx


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [47]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : ORGANISME → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [48]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans ID_ORGANISME : 15585123
Codes manquants dans ID_ORGANISME : 42840


**4) Similarité pour les non appariés (top 1 par clé)**

In [49]:
suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

⚠️ 2 libellés non appariés trouvés (total lignes concernées : 42840) :


,ORGANISME_NORM,COUNT
0,"min des mines, du petrole et de l energie",37766
1,"min delegue aupres du mae, charge int af et iv...",5074


,ORGANISME_NORM_NON_APPARIE,MATCH_LIBELLÉ_ORGANISME_NORM,MATCH_SCORE,LIBELLÉ_ORGANISME,ID_ORGANISME
0,"min des mines, du petrole et de l energie","min des mines , du petrole et de l energie",98,"Min. des Mines , du petrole et de l Energie",14
1,"min delegue aupres du mae, charge int af et iv...","min delegue aupres du pm, charge des sports et...",71,"Min. délégué aupres du PM, chargé des sports e...",29


**5a)Application MANUELLE (tu fournis tes choix)**

In [50]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "min des mines, du petrole et de l energie": "14"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 37766


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [51]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


🔁 Mapping enrichi : +1 lignes (net).


In [52]:
# liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [53]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['ID_ORGANISME_BEFORE'] != panel_solde_df['ID_ORGANISME']) | \
            (panel_solde_df['ID_ORGANISME_BEFORE'].isna() & panel_solde_df['ID_ORGANISME'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['ID_ORGANISME_BEFORE', 'ID_ORGANISME', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

Nombre d'observations modifiées ou complétées : 37766


,ID_ORGANISME_BEFORE,ID_ORGANISME,ORGANISME_NORM
789,<NA>,14,"min des mines, du petrole et de l energie"


In [54]:
# Supprimer CODE_ORGANISME_BEFORE
panel_solde_df=panel_solde_df.drop(columns=["ID_ORGANISME_BEFORE"])

In [55]:
# liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

## CLASSES ECHELONS

In [56]:
# Lecture du fichier Excel
fichier_ref = "grade"
df_ref = read_reference(bytes_io, fichier_ref)


=== grade | APERÇU BRUT (top 5) ===


,CODE_GRADE,LIBELLÉ_GRADE
0,10,Militaire
1,11,Classe Exceptionnelle 1er Echelon
2,12,Classe Exceptionnelle 2ème Echelon
3,13,Classe Exceptionnelle 3ème Echelon
4,15,Militaire


Colonnes BRUTES (grade) : ['CODE_GRADE', 'LIBELLÉ_GRADE']
Shape brut : (278, 2)

=== grade | APRÈS NETTOYAGE (top 5) ===


,CODE_GRADE,LIBELLÉ_GRADE
0,10,Militaire
1,11,Classe Exceptionnelle 1er Echelon
2,12,Classe Exceptionnelle 2ème Echelon
3,13,Classe Exceptionnelle 3ème Echelon
4,15,Militaire


Colonnes NETTOYÉES (grade) : ['CODE_GRADE', 'LIBELLÉ_GRADE']
Shape nettoyé : (278, 2)


In [57]:
# liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

**Paramètres**

In [58]:
nm_feuille = "ECHELONS" # nom de la feuille 
var_code_ext = "CODE_GRADE"
var_lib_ext  = "LIBELLÉ_GRADE"
var_cle_ext  = var_lib_ext

var_cle_panel  = "CLASSE/ECHELON"
var_code_panel = "CODE_GRADE"
var_lib_panel  = "CLASSE/ECHELON"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

**1) Vérification**

In [59]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 8 clés répétées, 54 lignes uniques concernées.


In [60]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 8 clés répétées, 54 lignes concernées.


,CODE_GRADE,LIBELLÉ_GRADE,LIBELLÉ_GRADE_NORM
0,10,Militaire,militaire
4,15,Militaire,militaire
5,16,Militaire,militaire
6,19,Militaire,militaire
11,24,Militaire,militaire
12,26,Militaire,militaire
13,27,Militaire,militaire
14,28,Militaire,militaire
15,29,Militaire,militaire
16,30,Militaire détaché,militaire detache


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [61]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : ECHELONS → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [62]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans CODE_GRADE : 15310755
Codes manquants dans CODE_GRADE : 317208


**4) Similarité pour les non appariés (top 1 par clé)**

In [63]:
suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

⚠️ 3 libellés non appariés trouvés (total lignes concernées : 317208) :


,CLASSE/ECHELON_NORM,COUNT
0,,317105
1,insp general 1er ech,90
2,adm general 2eme ech,13


,CLASSE/ECHELON_NORM_NON_APPARIE,MATCH_LIBELLÉ_GRADE_NORM,MATCH_SCORE,LIBELLÉ_GRADE,CODE_GRADE
1,insp general 1er ech,insp general 1er ch,97,Insp. Général 1er ch,I1
2,adm general 2eme ech,adm general 1er ech,87,Adm. Général 1er ech,A2
0,,AUCUNE_SUGGESTION,0,Aucune suggestion possible,None


**5a)Application MANUELLE (tu fournis tes choix)**

In [64]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    "insp general 1er ech": "I1"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 90


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [65]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


🔁 Mapping enrichi : +-1 lignes (net).


In [66]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [67]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_GRADE_BEFORE'] != panel_solde_df['CODE_GRADE']) | \
            (panel_solde_df['CODE_GRADE_BEFORE'].isna() & panel_solde_df['CODE_GRADE'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_GRADE_BEFORE', 'CODE_GRADE', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

Nombre d'observations modifiées ou complétées : 90


,CODE_GRADE_BEFORE,CODE_GRADE,CLASSE/ECHELON_NORM
384715,<NA>,I1,insp general 1er ech


In [68]:
# Suppression de la variable CODE_GRADE_BEFORE
panel_solde_df= panel_solde_df.drop(columns=["CODE_GRADE_BEFORE"])

In [69]:
# Renommer la variable pour éviter toute confusion 
panel_solde_df = panel_solde_df.rename(columns={"CODE_GRADE": "CODE_CLASSE/ECHELON"})

In [70]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [71]:
xlsx = pd.ExcelFile(bytes_io)
print("Feuilles disponibles :", xlsx.sheet_names)

Feuilles disponibles : ['lieu affectation', 'ORGANISME_OK', 'CODE_SITUATION_SOLDE', 'HISTORIQUE_ECHELLES_CORPS', 'ECHELLES_CORPS_ACTUEL', 'service', 'FONCTION', 'grade', 'LIBELLE POSTE']


## EMPLOIS

In [72]:
# Lecture du fichier Excel
fichier_ref = "HISTORIQUE_ECHELLES_CORPS"
df_ref = read_reference(bytes_io, fichier_ref)


=== HISTORIQUE_ECHELLES_CORPS | APERÇU BRUT (top 5) ===


,TABLEAU: LISTE EXHAUSTIVE DES CODIFICATIONS DES EMPLOIS DANS LA BASE DE DONNEES DE LA DIRECTION DE LA SOLDE,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,CODE_EMPLOI,LIBELLÉ_EMPLOI,GRADE_ADMINISTRATIF_ASSOCIE
3,C1AG,Assistant des Greffes et Parquets,C3
4,A2TE,Ingénieur des Techn. d'Elevage,A3


Colonnes BRUTES (HISTORIQUE_ECHELLES_CORPS) : ['TABLEAU: LISTE EXHAUSTIVE DES CODIFICATIONS  DES EMPLOIS DANS LA BASE DE DONNEES DE LA DIRECTION DE LA SOLDE', 'Unnamed: 1', 'Unnamed: 2']
Shape brut : (1258, 3)

=== HISTORIQUE_ECHELLES_CORPS | APRÈS NETTOYAGE (top 5) ===


,CODE_EMPLOI,LIBELLÉ_EMPLOI,GRADE_ADMINISTRATIF_ASSOCIE
0,C1AG,Assistant des Greffes et Parquets,C3
1,A2TE,Ingénieur des Techn. d'Elevage,A3
2,22I1,Inspec. Général: Educ. et Formation,A7
3,22I2,Inspec. Général: Ensgt Art. et Culturel,A7
4,22I3,Inspec. Général: Ensgt sous Sect. Social,A7


Colonnes NETTOYÉES (HISTORIQUE_ECHELLES_CORPS) : ['CODE_EMPLOI', 'LIBELLÉ_EMPLOI', 'GRADE_ADMINISTRATIF_ASSOCIE']
Shape nettoyé : (1255, 3)


In [73]:
# Lister les variables de la base panel
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

**Paramètres**

In [74]:
nm_feuille = "EMPLOIS" # nom de la feuille 
var_code_ext = "CODE_EMPLOI"
var_lib_ext  = "LIBELLÉ_EMPLOI"
var_cle_ext  = var_lib_ext

var_cle_panel  = "EMPLOI"
var_code_panel = "CODE_EMPLOI"
var_lib_panel  = "EMPLOI"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

**1) Vérification**

In [75]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 341 clés répétées, 761 lignes uniques concernées.


In [76]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 341 clés répétées, 761 lignes concernées.


,CODE_EMPLOI,LIBELLÉ_EMPLOI,LIBELLÉ_EMPLOI_NORM
0,C1AG,Assistant des Greffes et Parquets,assistant des greffes et parquets
1,A2TE,Ingénieur des Techn. d'Elevage,ingenieur des techn d elevage
7,23I1,Prof. Agrégé Educ. et Formation,prof agrege educ et formation
10,23I4,Prof. Agrégé Ensgt Tech. et Form. Prof.,prof agrege ensgt tech et form prof
12,24G1,Prof. Agrégé Educ. et Formation,prof agrege educ et formation
...,...,...,...
1239,86MD,Ingénieur des Médias,ingenieur des medias
1243,76EV,INGENIEUR PRINCIPAL: GENIE RURAL,ingenieur principal: genie rural
1244,88ET,INGENIEUR EN CHEF: ELECTROTECHNIQUE,ingenieur en chef: electrotechnique
1245,A1GO,Adm des sces financiers - Commerce,adm des sces financiers commerce


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [77]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : EMPLOIS → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [78]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans CODE_EMPLOI : 15312585
Codes manquants dans CODE_EMPLOI : 315378


**4) Similarité pour les non appariés (top 1 par clé)**

In [79]:
suggestions_best = best_suggestions(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_lib_ext=var_lib_ext, var_code_ext=var_code_ext,
    var_code_panel=var_code_panel,
    min_score_show=0
)

⚠️ 1 libellés non appariés trouvés (total lignes concernées : 315378) :


,EMPLOI_NORM,COUNT
0,,315378


,EMPLOI_NORM_NON_APPARIE,MATCH_LIBELLÉ_EMPLOI_NORM,MATCH_SCORE,LIBELLÉ_EMPLOI,CODE_EMPLOI
0,,AUCUNE_SUGGESTION,0,Aucune suggestion possible,None


**5a)Application MANUELLE (tu fournis tes choix)**

In [80]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    # ex: "tresorier charge d etudes g4": "797"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 0


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [81]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


INFO : rien à enrichir (pas de suggestions ou pas de choix).


In [82]:
# Liste des variables
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [83]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_EMPLOI_BEFORE'] != panel_solde_df['CODE_EMPLOI']) | \
            (panel_solde_df['CODE_EMPLOI_BEFORE'].isna() & panel_solde_df['CODE_EMPLOI'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_EMPLOI_BEFORE', 'CODE_EMPLOI', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)


Nombre d'observations modifiées ou complétées : 0


,CODE_EMPLOI_BEFORE,CODE_EMPLOI,EMPLOI_NORM


In [84]:
# Suppression de la variable CODE_EMPLOI_BEFORE
panel_solde_df=panel_solde_df.drop(columns=['CODE_EMPLOI_BEFORE'])

In [85]:
# Liste des variables pour vérification
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [86]:
xlsx = pd.ExcelFile(bytes_io)
print("Feuilles disponibles :", xlsx.sheet_names)

Feuilles disponibles : ['lieu affectation', 'ORGANISME_OK', 'CODE_SITUATION_SOLDE', 'HISTORIQUE_ECHELLES_CORPS', 'ECHELLES_CORPS_ACTUEL', 'service', 'FONCTION', 'grade', 'LIBELLE POSTE']


## SITUATION DANS L'ACTIVITE

In [87]:
# Lecture de la feuille ORGANISME du fichier de reférence (Excel)
fichier_ref = "CODE_SITUATION_SOLDE"
df_ref = read_reference(bytes_io, fichier_ref)


=== CODE_SITUATION_SOLDE | APERÇU BRUT (top 5) ===


,CODE_POSITION_SOLDE,LIBELLÉ_POSITION_SOLDE ( LIBELLE_SITUATION_SOLDE)
0,0,Annulé
1,1,Prise En Compte En Cours
2,9,Agent en Liquidation de Droits
3,10,En Activité
4,20,Suspendu Sans Solde


Colonnes BRUTES (CODE_SITUATION_SOLDE) : ['CODE_POSITION_SOLDE', 'LIBELLÉ_POSITION_SOLDE ( LIBELLE_SITUATION_SOLDE)']
Shape brut : (60, 2)

=== CODE_SITUATION_SOLDE | APRÈS NETTOYAGE (top 5) ===


,CODE_POSITION_SOLDE,LIBELLÉ_POSITION_SOLDE ( LIBELLE_SITUATION_SOLDE)
0,0,Annulé
1,1,Prise En Compte En Cours
2,9,Agent en Liquidation de Droits
3,10,En Activité
4,20,Suspendu Sans Solde


Colonnes NETTOYÉES (CODE_SITUATION_SOLDE) : ['CODE_POSITION_SOLDE', 'LIBELLÉ_POSITION_SOLDE ( LIBELLE_SITUATION_SOLDE)']
Shape nettoyé : (60, 2)


In [88]:
# Lister les variables de la base de données panel
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

**Paramètres**

In [89]:
nm_feuille = "STATUT_ACTIVITE" # nom de la feuille 
var_code_ext = "CODE_POSITION_SOLDE"
var_lib_ext  = "LIBELLÉ_POSITION_SOLDE ( LIBELLE_SITUATION_SOLDE)"
var_cle_ext  = var_lib_ext

var_cle_panel  = "SITUATION"
var_code_panel = "CODE_POSITION_SOLDE"
var_lib_panel  = "SITUATION"


# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"
var_norm_panel = f"{var_cle_panel}_NORM"

**1) Vérification**

In [90]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
✅ Aucun doublon détecté après normalisation.


In [91]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Aucun doublon détecté après normalisation.


**2) Mapping aléatoire (1 code / clé) + export dans un classeur commun**

In [92]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_3686/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : STATUT_ACTIVITE → Referentiels_unifies.xlsx


**3) Fusion dans le panel (panel_solde_df)**

In [93]:
# Utilisation de la fonction "merge_mapping_into_panel"
panel_solde_df, _, _ = merge_mapping_into_panel(
    panel_solde_df, mapping,
    var_norm_panel=var_norm_panel, var_norm_ext=var_norm_ext,
    var_code_panel=var_code_panel, var_code_ext=var_code_ext,
    var_lib_panel=var_lib_panel
)

Codes renseignés dans CODE_POSITION_SOLDE : 15627963
Codes manquants dans CODE_POSITION_SOLDE : 0


**5a)Application MANUELLE (tu fournis tes choix)**

In [94]:
# Dictionnaire des choix forcés : clé = libellé normalisé du panel, valeur = code du référentiel (fichier Excel) choisi
CHOIX_FORCES = {
    # "cle_norm_panel_exemple": "CODE_CHOISI"
    # ex: "tresorier charge d etudes g4": "797"
}
panel_solde_df, audit_manuel = apply_choices(
    panel_solde_df,
    var_code_panel=var_code_panel, var_norm_panel=var_norm_panel,
    choices_map=CHOIX_FORCES,
    var_cle_panel=var_cle_panel, var_lib_panel=var_lib_panel
)

✔ Lignes mises à jour : 0


**6a) Enrichir le mapping avec les choix retenus (manuel)**

In [95]:
#   on reconstruit le dict choices_map à partir des choix imposés manuellement
if not audit_manuel.empty:
    choices_map_manuel = dict(zip(
        panel_solde_df.loc[audit_manuel.index, var_norm_panel],
        panel_solde_df.loc[audit_manuel.index, var_code_panel]
    ))
else:
    choices_map_manuel = {}

mapping = enrich_mapping_with_choices(
    mapping_final=mapping,
    suggestions_best=suggestions_best,
    var_norm_panel=var_norm_panel,
    var_norm_ext=var_norm_ext, var_code_ext=var_code_ext, var_lib_ext=var_lib_ext,
    choices_map=choices_map_manuel
)


INFO : rien à enrichir (pas de suggestions ou pas de choix).


In [96]:
# Lister les variables de la base de données panel
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'nbre_date_naissances',
       'DATE_NAISSANCE_CORRIGEE', 'ANNEE_NAISSANCE_CORRIGEE',
       'MOIS_NAISSANCE_CORRIGEE', 'JOUR_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'SEXE_VALIDE', 'AGE_IMPUTE',
       'SITUATION_MATRIMONIALE_RECODE', 'NBRE_ENFTS_IMPUTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'ID_ORGANISME'

In [97]:
# Crée un masque pour repérer les lignes différentes ou où l'ancien code était vide et le nouveau est renseigné
mask_diff = (panel_solde_df['CODE_POSITION_SOLDE_BEFORE'] != panel_solde_df['CODE_POSITION_SOLDE']) | \
            (panel_solde_df['CODE_POSITION_SOLDE_BEFORE'].isna() & panel_solde_df['CODE_POSITION_SOLDE'].notna())

# Filtrer les lignes concernées
panel_solde_df_diff = panel_solde_df[mask_diff]

# Affiche le nombre d'observations uniques concernées
print(f"Nombre d'observations modifiées ou complétées : {len(panel_solde_df_diff)}")

# Garde uniquement les colonnes d'intérêt et supprime les doublons
panel_solde_df_diff_unique = panel_solde_df_diff[['CODE_POSITION_SOLDE_BEFORE', 'CODE_POSITION_SOLDE', var_norm_panel]].drop_duplicates()

# Affiche le résultat
display(panel_solde_df_diff_unique)

Nombre d'observations modifiées ou complétées : 0


,CODE_POSITION_SOLDE_BEFORE,CODE_POSITION_SOLDE,SITUATION_NORM


In [98]:
# Suppression de la variable CODE_POSITION_SOLDE_BEFORE
panel_solde_df=panel_solde_df.drop(columns=['CODE_POSITION_SOLDE_BEFORE'])

## SAUVEGARDE DU FICHIER DES CODES DEFINITIFS

In [104]:
# Connexion S3 / MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"
object_key = "Solde/Referentiels_unifies.xlsx"

# Lecture du fichier local ou en mémoire contenant toutes les feuilles
excel_path_local = "Referentiels_unifies.xlsx"  # ou un chemin local existant
all_sheets = pd.read_excel(excel_path_local, sheet_name=None)  # sheet_name=None -> toutes les feuilles

# Conversion en Excel en mémoire
excel_buffer = io.BytesIO()
with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
    for sheet_name, df in all_sheets.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)

# Upload sur S3
s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=excel_buffer.getvalue())

print(f"✅ Toutes les feuilles exportées sur S3 : s3://{bucket_name}/{object_key}")


✅ Toutes les feuilles exportées sur S3 : s3://admindataanstat/Solde/Referentiels_unifies.xlsx
